# Frequency Data Runs for the Olmo 2 Family
1B, 7B and 13B

Experiments are run on Colab. Change the model name variable and the revision for each separate run.

## Setup

In [34]:
# Imports
import sys
import torch
import time
from pathlib import Path

import importlib
import src

# Detect the RUNTIME, not a specific synced subfolder: a Colab sync may bring
# src/ without data/, so probing for /content/data is unreliable. Use the same
# signal Cell 0 uses (import google.colab).
#   Colab: repo is under /content
#   Local: notebook is in notebooks/, project root is one level up
try:
    import google.colab  # noqa: F401
    PROJECT_ROOT = Path('/content')
except ImportError:
    PROJECT_ROOT = Path.cwd().parent

# src/ must be importable under BOTH kernels (local + Colab).
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: /content


In [35]:
#  Device check
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: cuda


In [36]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "OLMo-2-1124-13B"
revision = "stage1-step99000-tokens831B"

tokenizer = AutoTokenizer.from_pretrained(
    f"allenai/{model_name}",
    revision=revision,
)

olmo = AutoModelForCausalLM.from_pretrained(
    f"allenai/{model_name}",
    revision=revision,
)

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/37.0k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

In [40]:
olmo = olmo.to("cuda")
olmo.eval()

print(next(olmo.parameters()).device)
print(next(olmo.parameters()).dtype)

OutOfMemoryError: CUDA out of memory. Tried to allocate 270.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 252.81 MiB is free. Including non-PyTorch memory, this process has 78.99 GiB memory in use. Of the allocated memory 78.49 GiB is allocated by PyTorch, and 13.05 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
olmo.eval()
olmo.config.use_cache = True

### Prompts

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", next(olmo.parameters()).device)
print("Model dtype:", next(olmo.parameters()).dtype)
print("Device map:", getattr(olmo, "hf_device_map", None))
print("Cache:", olmo.config.use_cache)

In [ ]:
from src.elicitation import run_all_prompts
class HFModelWrapper:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    @torch.inference_mode()
    def generate(self, prompt, max_new_tokens=50, temperature=0):
        device = next(self.model.parameters()).device

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
        ).to(device)

        output = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=self.tokenizer.eos_token_id,
        )

        return self.tokenizer.decode(
            output[0],
            skip_special_tokens=True,
        )


olmo_wrapped = HFModelWrapper(olmo, tokenizer)

olmo_wrapped = HFModelWrapper(olmo, tokenizer)
results_df = run_all_prompts(olmo_wrapped, model_name, PROJECT_ROOT)
results_df

In [ ]:
from src.frequency import build_frequency_table
dolma_freq = build_frequency_table(index="v4_olmo-mix-1124_llama")

In [ ]:
from src.accuracy_coding import code_response

results_df['accuracy'] = results_df.apply(
    lambda r: code_response(r['prompt_type'], r['concept'], r['prompt'], r['output']),
    axis=1
)

In [ ]:
score_map = {'correct': 1.0, 'partial': 0.5, 'incorrect': 0.0}
results_df['score'] = results_df['accuracy'].map(score_map)

compound_accuracy = results_df.groupby('concept').agg(
    mean_accuracy=('score', 'mean'),
    n=('score', 'count')
).reset_index()
compound_accuracy['compound'] = compound_accuracy['concept'].str.replace(' ', '_')
compound_accuracy['compound'] = compound_accuracy['compound'].str.lower()

In [ ]:
import numpy as np
from scipy.stats import spearmanr

merged = compound_accuracy.merge(dolma_freq, on='compound', how='inner')
merged['log_freq'] = np.log10(merged['bigram_count'].clip(lower=1))

rho, p = spearmanr(merged['log_freq'], merged['mean_accuracy'])
print(f"{model_name} Spearman: ρ={rho:.4f}, p={p:.4f}, n={len(merged)}")

In [ ]:
import numpy as np
from scipy.stats import spearmanr

dtype = next(olmo.parameters()).device

merged = compound_accuracy.merge(dolma_freq, on='compound', how='inner')
merged['log_freq'] = np.log10(merged['bigram_count'].clip(lower=1))

rho, p = spearmanr(merged['log_freq'], merged['mean_accuracy'])
print(f"{model_name} Spearman: ρ={rho:.4f}, p={p:.4f}, n={len(merged)}")

# Save results
merged.to_csv(f'{model_name.replace("/", "_")}_spearman_merged.csv', index=False)

with open(f'{model_name.replace("/", "_")}_spearman_result.md', 'w') as f:
  f.write(f"# Spearman Result: {model_name}\n")
  f.write(f"Revision: {revision}\n")
  f.write(f"Model dtype: {next(olmo.parameters()).dtype}\n")
  f.write(f"- ρ = {rho:.4f}\n")
  f.write(f"- p = {p:.4f}\n")
  f.write(f"- n = {len(merged)}\n")
  f.write(f"- corpus: OLMo-Mix-1124\n")
  f.write(f"- index: v4_olmo-mix-1124_llama\n\n")

print(f"Saved to {model_name.replace('/', '_')}_spearman_result.md")

In [ ]:
from pathlib import Path

output_path = Path(PROJECT_ROOT) / f"{run_id}-spearman.md"

with open(output_path, "w") as f:
    f.write(f"""# Spearman Result: {model_name}

- repository: allenai/{model_name}
- revision: {revision}
- dtype: {next(olmo.parameters()).dtype}
- device: {next(olmo.parameters()).device}
- rho: {rho:.4f}
- p: {p:.4g}
- n: {n}
""")

print(f"Saved to: {output_path.resolve()}")

In [ ]:
!zip -r olmo_experiment_bundle.zip \
    /content/*-results.csv \
    /content/*-spearman.md

### Delete Model & Clear Cache

In [42]:
# Free memory for next model
import gc
del olmo
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")
!rm -rf ~/.cache/huggingface

Memory cleared — GPU: 84.3GB allocated
